# 26 — Reprodução em ambiente isolado e supervisão da VM

Estado documental em **2026-10-29**. Esta etapa explica por que a reprodução entrou antes de Terraform, quais lacunas apareceram e quais aceites foram concluídos. As evidências nativas são saídas e capturas fornecidas pelo autor; os recibos operacionais permanecem na VM. Nenhuma célula deste notebook executa treino, inferência, instalação, kill ou reboot.

Referências principais: [roadmap](../../docs/project/ROADMAP.md), [acompanhamento da VM](../../docs/operations/VM_REPRODUCTION_PROGRESS.md), [supervisão](../../docs/operations/VM_USER_SUPERVISION.md) e [recibo documental](../../references/evidence/vm_reproduction_and_supervision_author_validation_2026-10-29.json).

## A pergunta antes de Terraform

O caminho completo funcionava no laboratório original. Era necessário demonstrar o que outra instalação precisaria produzir por conta própria. Terraform descreve infraestrutura como código (IaC), mas não elimina estados, credenciais, recibos, datasets ou processos que ficaram implícitos no laptop.

A escolha foi preservar o ambiente aceito e reproduzir o percurso numa VM isolada. Coexistência no mesmo Docker exigiria qualificação própria de nomes, volumes e ownership. A VM separou esses estados e revelou o contrato real da reprodução. O recorte se ampliou por dependências encontradas, não por exigir mais ferramentas.

## Ambientes e unidades de evidência

| Ambiente/recorte | O que foi observado | O que não transferimos |
| --- | --- | --- |
| Laboratório original | Jornada diária de 9.619 decisões/100 alertas e aceites históricos próprios | Baseline, credenciais, ownership e resultado de reboot não viram aceites da VM |
| Rodada `repro-v1` na VM | Dados, treino de três candidatos, Registry, API e rollback | Nova execução não prova igualdade de todos os modelos com o laboratório |
| Sessão `repro-arrival-200-v1` | 269.334 eventos, 200 decisões HTTP, 199 clientes, 13 alertas e 200 feedbacks nas decisões | Eventos de contexto e feedback não são inferências |
| Supervisão no host | Dois processos recuperados e métricas preservadas | Logout completo e reboot planejado aceitos posteriormente; não é restauração ou tolerância à perda do host |

As janelas já observadas continuam sendo demonstração operacional, sem nova confirmação independente de qualidade.

## O percurso reproduzido

1. Instalar ferramentas e criar fundação KinD/Helm com PostgreSQL/RustFS e tracking próprios.
2. Adquirir Bronze da origem; construir, publicar e restaurar Silver/Gold com verificadores.
3. Executar rodada, promover pela política existente e verificar pesos da API e rollback.
4. Produzir a fonte causal e reconciliar uma sessão pela API do champion da VM.
5. Adaptar a exportação da sessão sem fabricar recibos do replay legado.
6. Instalar observabilidade somente sessão e os quatro dashboards complementares.
7. Revisar os painéis, consolidar processos do host e ensaiar sua recuperação.

Não chamamos esse percurso de instalador integral aprovado: ainda faltam aceites do portal e um gate que parta do checkout vazio e encerre todos os pré-requisitos de forma repetível.

## O que o Grafana pode afirmar nesta sessão

O painel original e os quatro componentes usam a mesma identidade de sessão/modelo/release. Receiver, Prometheus/Grafana e dois volumes persistentes são reaproveitados. A coleta expira após 30 segundos sem publicação.

As 200 requisições permitem conferir saúde das features e feedback. Não sustentam duas janelas independentes de 1.000 decisões; drift permanece sem suporte. Latência HTTP p95 de 44,8 ms vem de recibos da sessão, não mede espera em fila nem atraso de emissão. Esses tempos não medidos ficam indisponíveis.

A coorte contém duas transações fraudulentas e dois cliente-dias positivos: um capturado e um perdido. Precisão de 7,69%, recall de 50% e AP (Average Precision) transacional de 0,5058479532 são resultados dessa coorte, não avaliação global. Os 13 alertas são de triagem, não firing de alertas de monitoramento.

## Por que os testes verdes anteriores não bastaram

Tox executava contratos, cálculos e checks num ambiente Python isolado. Não isolava automaticamente Docker, Kubernetes, volumes e recibos do host. Os ensaios nativos também partiam de ambientes preparados. Faltava o gate integral de checkout vazio.

Duas classes de falhas apareceram: fixtures que liam o perfil operacional real da VM e fixtures históricas que omitiam transições novas ou misturavam hashes antigos com bytes atuais. As 15 falhas dos CIs anteriores eram falhas de testes, não timeout. A correção preservou comparações exatas e a rejeição de alteração científica.

A preparação da correção passou com 1.852 testes, 59 skips, 166 warnings e 12 subtests. O CI #70 aprovou os três jobs do PR #13, integrado em `66e2bbf`. A supervisão veio depois e foi integrada no PR #14 em `9d9c8f2`; o autor informou CI aprovado nesse PR. Não apresentar os skips como testes aprovados nem usar o CI anterior para validar código posterior.

## Responsabilidades no host e no cluster

| Componente | Supervisor/forma de acesso |
| --- | --- |
| API, tracking, armazenamento e monitoramento no cluster | Kubernetes/Helm e seus volumes/recibos |
| Projeção de agregados da sessão | `fraud-session-observer.service`; observador existente com lock |
| Grafana no loopback da VM, porta 13001 | `fraud-grafana-access.service`; acesso foreground com probe HTTP |
| Navegador no Alienware, porta 13005 no ensaio | Túnel SSH separado; não é supervisor dos workloads |

O processo pai do acesso podia continuar vivo com port-forward morto. Por isso, a unidade do Grafana precisa verificar saúde, além de reiniciar após saída do processo. O lock impede dois observadores; não se apaga o lock para iniciar outro.

## Diagnósticos que mudaram a operação

A troca de pods invalidou encaminhamentos antigos. O login exigiu distinguir a instância da VM e sua credencial da instalação original; autenticação direta foi conferida antes de redefinir qualquer senha. Os painéis de features exigiram uma única tabela por consulta, com Feature | Valor, em vez de um seletor de séries e metadados técnicos.

No handoff para supervisão, a porta 13001 estava sem listener, mas tinha conexão TIME-WAIT. O probe de bind foi corrigido com reutilização de endereço, mantendo a recusa de listener ativo. Foram acrescentados testes reais de socket; não se alterou sysctl nem se matou processo desconhecido para liberar a porta.

## Ensaio nativo de recuperação dos processos

O autor informou instalação de unidades sem ativação inicial, troca explícita dos processos, check da mesma sessão e ensaio de falha controlada. O resultado foi `vm_user_service_recovery_verified`:

| Unidade | PID inicial | PID após ensaio | Check posterior |
| --- | ---: | ---: | --- |
| Observador | 506569 | 512559 | Mesmo PID, NRestarts=1 |
| Acesso Grafana | 506570 | 514819 | Mesmo PID, NRestarts=1 e HTTP saudável |

As contagens e os hashes aceitos foram preservados. Não houve novas inferências. Esses PIDs documentam a execução; não devem ser copiados para comandos futuros de encerramento.

Posteriormente, o autor forneceu Linger=yes e confirmou o encerramento de todas as conexões SSH, túneis e sessão de console. O check após reconexão manteve PIDs 512559/514819 e NRestarts=1, com hashes e contagens preservados: persistência após logout aceita.

No reboot planejado, o boot ID mudou de `6aebb2b3-58fc-441e-ad0e-badeecbbb5bb` para `24524ae4-b9a4-464b-8f1a-a1319ffc2acd`. Observador PID 4455 e acesso Grafana PID 5026 ficaram active/running, ambos NRestarts=1, HTTP saudável e mesma sessão. O journal mostra falha inicial de kubectl e recuperação automática: observador às 15:33:46 e Grafana pronto às 15:33:51 (horários do guest, offset não informado). A hipótese de cluster ainda iniciando é compatível com a sequência, sem causa exata exposta.

Os flags originais false não foram reescritos. A validação externa combina confirmação, boot IDs, check e logs; não comprova restauração de backup ou tolerância à perda do host. [Recibo complementar](../../references/evidence/vm_logout_reboot_author_validation_2026-10-29.json).

## Critérios para encerrar a etapa e avançar

O [roadmap](../../docs/project/ROADMAP.md) mantém as pendências: concluir inventário/acessos, fechar o portal da VM, qualificar instalação integral e planejar ciclo completo de parada/subida e backup/restauração. Supervisão integrada, logout completo e reboot planejado aceitos no recorte documentado.

A configuração de Linger possibilita persistência do user manager; sua presença não é resultado de ensaio. A VM de um nó também não demonstra tolerância à perda do host.

Terraform vem depois do fluxo estabilizado, com plano/apply e segundo plan sem mudanças. Avaliar coordenação dos pipelines é outro recorte; Prefect, Airflow, broker e feature store não são requisitos automáticos. Nuvem segue opcional e independente da entrega local.

## Consultar o estado, sem iniciar outro observador

Na VM com o código e as unidades instalados:

```bash
poetry run python -m scripts.platform.host_supervision status
journalctl --user -u fraud-session-observer.service -u fraud-grafana-access.service -n 60 --no-pager
```

Os comandos completos de instalação, troca de processos, check e recuperação ficam no [runbook](../../docs/operations/VM_USER_SUPERVISION.md). O notebook registra decisões e resultados; não substitui esse procedimento nem dispara os ensaios ao ser aberto.